# Previsão de CaO Livre com Redes Neurais Avançadas
Este notebook utiliza abordagens de Deep Learning (LSTM, TCN) para prever a concentração de CaO Livre.

**Melhorias Implementadas (Baseadas em Análise de Performance):**
1. **Informação Mútua (Mutual Information):** Realiza a varredura dinâmica para encontrar e aplicar o lag ótimo de cada variável de processo em relação à variável alvo, alinhando fisicamente a causa e o efeito.
2. **Janelas Deslizantes Curtas ($SEQ\_LEN$):** Com as features já alinhadas pelo lag, passamos apenas os últimos 15 minutos para a rede capturar as dinâmicas e derivadas recentes (ex: taxa de queda da temperatura).
3. **Early Stopping e Validação:** Separação explícita de Treino, Validação e Teste. O modelo interrompe o treinamento automaticamente quando a métrica de validação para de cair, evitando o *overfitting*.
4. **ReduceLROnPlateau:** O otimizador ajusta dinamicamente a taxa de aprendizado ao chegar perto dos mínimos globais.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt
import copy
import warnings
warnings.filterwarnings("ignore")

## Carregamento e Pré-Processamento

In [ ]:
DIR_DATA = "./data/"
base_name = "CaO Livre.csv"
timestamp = "Timestamp"
target_col = "AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS"
target_limpo = target_col + "_CLEAN"

df_dataset = pd.read_csv(DIR_DATA + base_name, sep=";", decimal=".", encoding="utf-8-sig")
df_dataset[timestamp] = pd.to_datetime(df_dataset[timestamp], format="%Y-%m-%d %H:%M:%S")
df_dataset = df_dataset.set_index(timestamp)
df_dataset = df_dataset.resample("1min").asfreq()
df_dataset = df_dataset.reset_index()

for col in df_dataset.columns:
    if col != "Timestamp":
        df_dataset[col] = pd.to_numeric(df_dataset[col], errors="coerce")

start_date = pd.to_datetime("2026-01-01 00:00:00")
end_date = pd.to_datetime("2026-08-01 00:00:00")
mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_dataset = df_dataset.loc[mask]
df_dataset.set_index(timestamp, inplace=True)
df_dataset.ffill(inplace=True)

## Remoção de Paradas de Forno e Deplato

In [ ]:
mask_parada = df_dataset["AR.F1.II12501"] < 200
features_para_limpar = [col for col in df_dataset.columns if col != timestamp]
df_dataset.loc[mask_parada, features_para_limpar] = np.nan

# Deplato
_s = df_dataset[target_col]
_run_id = _s.ne(_s.shift()).cumsum()
_run_len = _s.groupby(_run_id).transform("size")
_eh_medicao = (_run_len >= 10) & _s.notna() & (_run_id != _run_id.shift())
df_dataset[target_limpo] = _s.where(_eh_medicao, np.nan)

## Seleção de Lags por Informação Mútua
Varre lags de 0 a 180 min e mantém o de maior informação mútua com o CaO Livre.

In [ ]:
max_lag = 180
features = [col for col in df_dataset.columns if col not in [target_col, target_limpo]]
df_features_filled = df_dataset[features].ffill().bfill()

# Definição de corte (usando os 70% iniciais das medicoes validas para calcular os lags sem vazamento)
_instantes_lab = df_dataset[target_limpo].dropna().index
_cut = max(1, int(len(_instantes_lab) * 0.70))
train_cutoff_date = _instantes_lab[_cut - 1]

y_target = df_dataset[target_limpo]
indices_validos = y_target.dropna().index
indices_validos = indices_validos[indices_validos <= train_cutoff_date]
y_valid = y_target.loc[indices_validos]

mi_matrix = np.zeros((len(features), max_lag + 1))

print("Calculando Informação Mútua por lag (isso pode levar alguns minutos)...")
for lag in range(max_lag + 1):
    shifted_features = df_features_filled.shift(lag)
    X_valid = shifted_features.loc[indices_validos]
    X_valid = X_valid.bfill().fillna(0)
    mi_scores = mutual_info_regression(X_valid, y_valid, random_state=42)
    mi_matrix[:, lag] = mi_scores

df_lags_mi = pd.DataFrame(mi_matrix, index=features, columns=range(max_lag + 1))
df_tabela_mi = df_lags_mi.unstack().reset_index()
df_tabela_mi.columns = ['Lag (min)', 'Variável', 'Informação Mútua']
df_tabela_mi = df_tabela_mi.sort_values(by='Informação Mútua', ascending=False)
df_tabela_mi = df_tabela_mi.drop_duplicates(subset=['Variável'], keep='first').reset_index(drop=True)

X_features = pd.DataFrame(index=df_dataset.index)
for _, row in df_tabela_mi.iterrows():
    var_nome = row['Variável']
    lag_otimo = int(row['Lag (min)'])
    col_name = f"{var_nome}_lag{lag_otimo}"
    X_features[col_name] = df_dataset[var_nome].shift(lag_otimo)

X_features["AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS_lag1"] = df_dataset[target_limpo].ffill().shift(1)
X_features = X_features.bfill().fillna(0)

## Construção de Sequências e Divisão de Treino / Validação / Teste
Ao isolar uma fatia de Validação (10%), podemos aplicar **Early Stopping** para evitar Overfitting.
O Tamanho da sequência de entrada da rede (`SEQ_LEN`) dita quantos minutos no passado alinhado a rede olha.

In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))
X_scaled = scaler.fit_transform(X_features)
X_scaled_df = pd.DataFrame(X_scaled, index=X_features.index, columns=X_features.columns)

SEQ_LEN = 1 # Parâmetro ajustável! Tente 5, 10 ou 20 minutos de histórico das derivadas.

X_seq, y_seq, ts_validos = [], [], []

for i, ts in enumerate(df_dataset.index):
    if pd.notna(df_dataset[target_limpo].iloc[i]):
        if i >= SEQ_LEN - 1:
            seq = X_scaled_df.iloc[i - SEQ_LEN + 1 : i + 1].values
            if not np.isnan(seq).any():
                X_seq.append(seq)
                y_seq.append(df_dataset[target_limpo].iloc[i])
                ts_validos.append(ts)

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)
ts_validos = pd.DatetimeIndex(ts_validos)

# 70% Treino | 10% Validacao | 20% Teste
_cut_train = max(1, int(len(ts_validos) * 0.70))
_cut_val   = max(1, int(len(ts_validos) * 0.80))
cutoff_train = ts_validos[_cut_train - 1]
cutoff_val   = ts_validos[_cut_val - 1]

train_mask = ts_validos <= cutoff_train
val_mask   = (ts_validos > cutoff_train) & (ts_validos <= cutoff_val)
test_mask  = ts_validos > cutoff_val

X_train, y_train = X_seq[train_mask], y_seq[train_mask]
X_val, y_val     = X_seq[val_mask], y_seq[val_mask]
X_test, y_test   = X_seq[test_mask], y_seq[test_mask]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32).view(-1, 1))
val_dataset   = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32).view(-1, 1))
test_dataset  = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32).view(-1, 1))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Função de Treinamento com Early Stopping e LRScheduler

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100, patience=15):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x_batch.size(0)
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                y_pred = model(x_batch)
                loss = criterion(y_pred, y_batch)
                val_loss += loss.item() * x_batch.size(0)
        val_loss /= len(val_loader.dataset)
        
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if (epoch + 1) % 10 == 0 or epochs_no_improve == patience:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
            
        if epochs_no_improve >= patience:
            print(f"Early stopping trigado na época {epoch+1}")
            break
            
    model.load_state_dict(best_model_wts)
    return model

## Modelo LSTM

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

print("Treinando LSTM...")
model_lstm = LSTMModel(input_size=X_train.shape[2])
criterion = nn.MSELoss()
optimizer_lstm = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

model_lstm = train_model(model_lstm, train_loader, val_loader, criterion, optimizer_lstm, epochs=150, patience=20)

## Modelo CNN (Temporal Convolutional Network 1D)
TCNs normalmente são mais velozes e se saem melhor ignorando ruídos (blips) em indústrias.

In [ ]:
class TCNModel(nn.Module):
    def __init__(self, input_size, num_channels=64):
        super(TCNModel, self).__init__()
        self.conv1 = nn.Conv1d(input_size, num_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(num_channels, num_channels, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(num_channels, 1)
        
    def forward(self, x):
        # CNN 1D espera formato (batch_size, num_channels, seq_len)
        x = x.transpose(1, 2)
        out = self.relu(self.conv1(x))
        out = self.relu(self.conv2(out))
        out = self.pool(out).squeeze(2)
        out = self.fc(out)
        return out

print("\nTreinando TCN...")
model_tcn = TCNModel(input_size=X_train.shape[2])
optimizer_tcn = torch.optim.Adam(model_tcn.parameters(), lr=0.001)

model_tcn = train_model(model_tcn, train_loader, val_loader, criterion, optimizer_tcn, epochs=150, patience=20)

## Avaliação dos Modelos e R² na Base de Teste

In [ ]:
def evaluate_model(model, name):
    model.eval()
    with torch.no_grad():
        y_pred_train = model(torch.tensor(X_train, dtype=torch.float32)).numpy().flatten()
        y_pred_test = model(torch.tensor(X_test, dtype=torch.float32)).numpy().flatten()

    r2_tr = r2_score(y_train, y_pred_train)
    r2_te = r2_score(y_test, y_pred_test)
    rmse_te = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_te = mean_absolute_error(y_test, y_pred_test)
    
    print(f"=== {name} ===")
    print(f"[Treino] R2: {r2_tr:.4f}")
    print(f"[Teste ] R2: {r2_te:.4f} | RMSE: {rmse_te:.4f} | MAE: {mae_te:.4f}\n")
    
    return y_pred_test

preds_lstm = evaluate_model(model_lstm, "LSTM")
preds_tcn = evaluate_model(model_tcn, "TCN (CNN 1D)")

## Visualização dos Resultados

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(ts_validos[test_mask], y_test, label="Real (Laboratório)", color="black", marker="o", linestyle="-", alpha=0.6)
plt.plot(ts_validos[test_mask], preds_lstm, label="Previsão LSTM", color="orange", marker="s", linestyle="--", alpha=0.8)
plt.plot(ts_validos[test_mask], preds_tcn, label="Previsão TCN", color="blue", marker="x", linestyle="-.", alpha=0.8)
plt.title("CaO Livre: Real vs Previsão (Base de Teste Out-of-Sample)")
plt.xlabel("Tempo")
plt.ylabel("CaO Livre")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()